# NLBSE 2027 Skill Classification — Current Dataset ExperimentsThis cleaned copy keeps the experiments using `skillscope_data_v2.db` (3,070 issue rows) and removes the earlier notebook experiments on a different database snapshot.The reported split is a team-defined random 80/20 holdout (`random_state=42`). It is documented as the split used in these experiments, not as an organizer-provided assignment.Best observed setting in this notebook: word + character TF-IDF, One-vs-Rest Logistic Regression (`C=3`), threshold `0.26`; validation micro-precision `0.5855`, micro-recall `0.6294`, micro-F1 `0.6067`.

In [ ]:
from pathlib import Path
import sqlite3

db_path = Path("/content/skillscope_data_v2.db")

if not db_path.exists():
    print("Database file not found in this Colab session.")
else:
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM nlbse_tool_competition_data_by_issue
    """)
    row_count = cursor.fetchone()[0]

    cursor.execute("""
        PRAGMA table_info(nlbse_tool_competition_data_by_issue)
    """)
    column_names = [row[1] for row in cursor.fetchall()]

    print("Database file found:", db_path)
    print("Rows:", row_count)
    print("Columns:", len(column_names))
    print("First 12 columns:", column_names[:12])
    print("Last 5 columns:", column_names[-5:])

    connection.close()

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from huggingface_hub import hf_hub_download

zip_path = hf_hub_download(
    repo_id="NLBSE/SkillCompetition",
    filename="skillscope_data_v2.zip",
    repo_type="dataset"
)

with ZipFile(zip_path, "r") as archive:
    archive.extract("skillscope_data_v2.db", "/content")

db_path = Path("/content/skillscope_data_v2.db")

print("Database extracted:", db_path.exists())
if db_path.exists():
    print("Database size (MB):", round(db_path.stat().st_size / (1024 * 1024), 2))

In [ ]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("/content/skillscope_data_v2.db")

data = pd.read_sql_query(
    "SELECT * FROM nlbse_tool_competition_data_by_issue",
    connection
)

connection.close()

metadata_columns = list(data.columns[:9])
label_columns = list(data.columns[9:])

positive_counts = (data[label_columns] > 0).sum()

print("Dataset shape:", data.shape)
print("Input metadata columns:", metadata_columns)
print("Number of skill labels:", len(label_columns))
print("Labels with no positive examples:", int((positive_counts == 0).sum()))
print("Rows missing issue text:", int(data["issue text"].isna().sum()))
print("Rows missing issue description:", int(data["issue description"].isna().sum()))
print("Is 'Test' a skill label?", "Test" in label_columns)

In [ ]:
from sklearn.model_selection import train_test_split

active_label_columns = [
    column for column in label_columns
    if (data[column] > 0).any()
]

X_text = (
    data["issue text"].fillna("").astype(str)
    + " "
    + data["issue description"].fillna("").astype(str)
)

Y = (data[active_label_columns] > 0).astype("int8")

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_text,
    Y,
    test_size=0.20,
    random_state=42
)

labels_missing_from_training = int((y_train.sum(axis=0) == 0).sum())

print("Split type: provisional random 80/20 split")
print("Training rows:", len(X_train_text))
print("Validation rows:", len(X_val_text))
print("Labels used:", len(active_label_columns))
print("Labels with no positive training examples:", labels_missing_from_training)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

word_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=15000,
    min_df=2,
    sublinear_tf=True
)

X_train_word = word_vectorizer.fit_transform(X_train_text)
X_val_word = word_vectorizer.transform(X_val_text)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)

X_train = hstack([X_train_word, X_train_char]).tocsr()
X_val = hstack([X_val_word, X_val_char]).tocsr()

print("Training feature matrix:", X_train.shape)
print("Validation feature matrix:", X_val.shape)

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

model = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, solver="liblinear"),
    n_jobs=-1
)

model.fit(X_train, y_train)

val_probabilities = model.predict_proba(X_val)
val_predictions = (val_probabilities >= 0.50).astype("int8")

print("Validation Micro-Precision:",
      round(precision_score(y_val, val_predictions, average="micro", zero_division=0), 4))
print("Validation Micro-Recall:",
      round(recall_score(y_val, val_predictions, average="micro", zero_division=0), 4))
print("Validation Micro-F1:",
      round(f1_score(y_val, val_predictions, average="micro", zero_division=0), 4))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

best_result = None

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    predictions = (val_probabilities >= threshold).astype("int8")

    precision = precision_score(
        y_val, predictions, average="micro", zero_division=0
    )
    recall = recall_score(
        y_val, predictions, average="micro", zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, average="micro", zero_division=0
    )

    print(
        f"Threshold={threshold:.2f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | Micro-F1={f1:.4f}"
    )

    if best_result is None or f1 > best_result[3]:
        best_result = (threshold, precision, recall, f1)

print("\nBest threshold:", best_result[0])
print("Best Micro-Precision:", round(best_result[1], 4))
print("Best Micro-Recall:", round(best_result[2], 4))
print("Best Micro-F1:", round(best_result[3], 4))

In [ ]:
import pandas as pd
from google.colab import files

experiment = pd.DataFrame([{
    "Model": "Word + Character TF-IDF + Logistic Regression",
    "Split": "Provisional random 80/20, random_state=42",
    "Threshold": best_result[0],
    "Micro-Precision": best_result[1],
    "Micro-Recall": best_result[2],
    "Micro-F1": best_result[3]
}])

experiment.to_csv("provisional_experiment_result.csv", index=False)
files.download("provisional_experiment_result.csv")

In [ ]:
word_only_model = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, solver="liblinear"),
    n_jobs=-1
)

word_only_model.fit(X_train_word, y_train)
word_only_probabilities = word_only_model.predict_proba(X_val_word)

word_only_best = None

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    predictions = (word_only_probabilities >= threshold).astype("int8")

    precision = precision_score(
        y_val, predictions, average="micro", zero_division=0
    )
    recall = recall_score(
        y_val, predictions, average="micro", zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, average="micro", zero_division=0
    )

    print(
        f"Threshold={threshold:.2f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | Micro-F1={f1:.4f}"
    )

    if word_only_best is None or f1 > word_only_best[3]:
        word_only_best = (threshold, precision, recall, f1)

print("\nBest word-only threshold:", word_only_best[0])
print("Best word-only Micro-Precision:", round(word_only_best[1], 4))
print("Best word-only Micro-Recall:", round(word_only_best[2], 4))
print("Best word-only Micro-F1:", round(word_only_best[3], 4))

In [ ]:
char_only_model = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, solver="liblinear"),
    n_jobs=-1
)

char_only_model.fit(X_train_char, y_train)
char_only_probabilities = char_only_model.predict_proba(X_val_char)

char_only_best = None

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    predictions = (char_only_probabilities >= threshold).astype("int8")

    precision = precision_score(
        y_val, predictions, average="micro", zero_division=0
    )
    recall = recall_score(
        y_val, predictions, average="micro", zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, average="micro", zero_division=0
    )

    print(
        f"Threshold={threshold:.2f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | Micro-F1={f1:.4f}"
    )

    if char_only_best is None or f1 > char_only_best[3]:
        char_only_best = (threshold, precision, recall, f1)

print("\nBest character-only threshold:", char_only_best[0])
print("Best character-only Micro-Precision:", round(char_only_best[1], 4))
print("Best character-only Micro-Recall:", round(char_only_best[2], 4))
print("Best character-only Micro-F1:", round(char_only_best[3], 4))

In [ ]:
import pandas as pd
from google.colab import files

comparison = pd.DataFrame([
    {
        "Model": "Word + Character TF-IDF",
        "Threshold": best_result[0],
        "Micro-Precision": best_result[1],
        "Micro-Recall": best_result[2],
        "Micro-F1": best_result[3]
    },
    {
        "Model": "Word-only TF-IDF",
        "Threshold": word_only_best[0],
        "Micro-Precision": word_only_best[1],
        "Micro-Recall": word_only_best[2],
        "Micro-F1": word_only_best[3]
    },
    {
        "Model": "Character-only TF-IDF",
        "Threshold": char_only_best[0],
        "Micro-Precision": char_only_best[1],
        "Micro-Recall": char_only_best[2],
        "Micro-F1": char_only_best[3]
    }
])

comparison["Split"] = "Provisional random 80/20, random_state=42"
comparison.to_csv("model_comparison_provisional.csv", index=False)

print(comparison.to_string(index=False))
files.download("model_comparison_provisional.csv")

In [ ]:
for c_value in [0.1, 3.0]:
    candidate_model = OneVsRestClassifier(
        LogisticRegression(
            C=c_value,
            max_iter=1000,
            solver="liblinear"
        ),
        n_jobs=-1
    )

    candidate_model.fit(X_train, y_train)
    probabilities = candidate_model.predict_proba(X_val)
    predictions = (probabilities >= 0.25).astype("int8")

    precision = precision_score(
        y_val, predictions, average="micro", zero_division=0
    )
    recall = recall_score(
        y_val, predictions, average="micro", zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, average="micro", zero_division=0
    )

    print(
        f"C={c_value} | Threshold=0.25 | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | Micro-F1={f1:.4f}"
    )

In [ ]:
c_value = 10.0

candidate_model = OneVsRestClassifier(
    LogisticRegression(
        C=c_value,
        max_iter=1000,
        solver="liblinear"
    ),
    n_jobs=-1
)

candidate_model.fit(X_train, y_train)
probabilities = candidate_model.predict_proba(X_val)
predictions = (probabilities >= 0.25).astype("int8")

precision = precision_score(
    y_val, predictions, average="micro", zero_division=0
)
recall = recall_score(
    y_val, predictions, average="micro", zero_division=0
)
f1 = f1_score(
    y_val, predictions, average="micro", zero_division=0
)

print(f"C={c_value} | Threshold=0.25")
print("Micro-Precision:", round(precision, 4))
print("Micro-Recall:", round(recall, 4))
print("Micro-F1:", round(f1, 4))

In [26]:
import pandas as pd
from google.colab import files

updated_results = pd.DataFrame([
    {
        "Model": "Word + Character TF-IDF, C=1",
        "Threshold": 0.25,
        "Micro-Precision": 0.570766,
        "Micro-Recall": 0.641198,
        "Micro-F1": 0.603936
    },
    {
        "Model": "Word + Character TF-IDF, C=0.1",
        "Threshold": 0.25,
        "Micro-Precision": 0.5125,
        "Micro-Recall": 0.5637,
        "Micro-F1": 0.5369
    },
    {
        "Model": "Word + Character TF-IDF, C=3",
        "Threshold": 0.25,
        "Micro-Precision": 0.5769,
        "Micro-Recall": 0.6385,
        "Micro-F1": 0.6062
    },
    {
        "Model": "Word + Character TF-IDF, C=10",
        "Threshold": 0.25,
        "Micro-Precision": 0.5855,
        "Micro-Recall": 0.6173,
        "Micro-F1": 0.6010
    },
    {
        "Model": "Word-only TF-IDF",
        "Threshold": word_only_best[0],
        "Micro-Precision": word_only_best[1],
        "Micro-Recall": word_only_best[2],
        "Micro-F1": word_only_best[3]
    },
    {
        "Model": "Character-only TF-IDF",
        "Threshold": char_only_best[0],
        "Micro-Precision": char_only_best[1],
        "Micro-Recall": char_only_best[2],
        "Micro-F1": char_only_best[3]
    }
])

updated_results["Split"] = "Provisional random 80/20, random_state=42"
updated_results.to_csv("updated_experiment_comparison.csv", index=False)

print(updated_results.to_string(index=False))
files.download("updated_experiment_comparison.csv")

                         Model  Threshold  Micro-Precision  Micro-Recall  Micro-F1                                     Split
  Word + Character TF-IDF, C=1       0.25         0.570766      0.641198  0.603936 Provisional random 80/20, random_state=42
Word + Character TF-IDF, C=0.1       0.25         0.512500      0.563700  0.536900 Provisional random 80/20, random_state=42
  Word + Character TF-IDF, C=3       0.25         0.576900      0.638500  0.606200 Provisional random 80/20, random_state=42
 Word + Character TF-IDF, C=10       0.25         0.585500      0.617300  0.601000 Provisional random 80/20, random_state=42
              Word-only TF-IDF       0.25         0.545595      0.613631  0.577617 Provisional random 80/20, random_state=42
         Character-only TF-IDF       0.25         0.559777      0.631647  0.593544 Provisional random 80/20, random_state=42


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
best_c3_model = OneVsRestClassifier(
    LogisticRegression(C=3.0, max_iter=1000, solver="liblinear"),
    n_jobs=-1
)

best_c3_model.fit(X_train, y_train)
c3_probabilities = best_c3_model.predict_proba(X_val)

fine_threshold_results = []

for threshold in [0.20, 0.21, 0.22, 0.23, 0.24, 0.25,
                  0.26, 0.27, 0.28, 0.29, 0.30]:
    predictions = (c3_probabilities >= threshold).astype("int8")

    precision = precision_score(
        y_val, predictions, average="micro", zero_division=0
    )
    recall = recall_score(
        y_val, predictions, average="micro", zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, average="micro", zero_division=0
    )

    fine_threshold_results.append(
        (threshold, precision, recall, f1)
    )

best_c3_result = max(fine_threshold_results, key=lambda result: result[3])

for threshold, precision, recall, f1 in fine_threshold_results:
    print(
        f"Threshold={threshold:.2f} | "
        f"Precision={precision:.4f} | Recall={recall:.4f} | "
        f"Micro-F1={f1:.4f}"
    )

print("\nBest C=3 threshold:", best_c3_result[0])
print("Precision:", round(best_c3_result[1], 4))
print("Recall:", round(best_c3_result[2], 4))
print("Micro-F1:", round(best_c3_result[3], 4))

Threshold=0.20 | Precision=0.5294 | Recall=0.6857 | Micro-F1=0.5975
Threshold=0.21 | Precision=0.5388 | Recall=0.6761 | Micro-F1=0.5997
Threshold=0.22 | Precision=0.5492 | Recall=0.6668 | Micro-F1=0.6023
Threshold=0.23 | Precision=0.5586 | Recall=0.6553 | Micro-F1=0.6031
Threshold=0.24 | Precision=0.5687 | Recall=0.6478 | Micro-F1=0.6057
Threshold=0.25 | Precision=0.5769 | Recall=0.6385 | Micro-F1=0.6062
Threshold=0.26 | Precision=0.5855 | Recall=0.6294 | Micro-F1=0.6067
Threshold=0.27 | Precision=0.5914 | Recall=0.6204 | Micro-F1=0.6055
Threshold=0.28 | Precision=0.5999 | Recall=0.6134 | Micro-F1=0.6066
Threshold=0.29 | Precision=0.6055 | Recall=0.6030 | Micro-F1=0.6042
Threshold=0.30 | Precision=0.6136 | Recall=0.5957 | Micro-F1=0.6045

Best C=3 threshold: 0.26
Precision: 0.5855
Recall: 0.6294
Micro-F1: 0.6067


In [32]:
import joblib
from google.colab import files

final_bundle = {
    "model": best_c3_model,
    "word_vectorizer": word_vectorizer,
    "char_vectorizer": char_vectorizer,
    "label_columns": active_label_columns,
    "threshold": best_c3_result[0],
    "split": "Random 80/20, random_state=42",
    "validation_metrics": {
        "micro_precision": best_c3_result[1],
        "micro_recall": best_c3_result[2],
        "micro_f1": best_c3_result[3]
    }
}

joblib.dump(final_bundle, "nlbse_best_model_provisional.joblib")
files.download("nlbse_best_model_provisional.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>